# Batch Processing with LandmarkDiff

This notebook demonstrates how to process multiple face images in batch,
apply surgical procedure predictions, and collect results for analysis.

**Requirements:**
- `pip install landmarkdiff`
- A directory of face images (JPG/PNG)
- For GPU inference: CUDA-capable GPU with PyTorch

In [ ]:
from pathlib import Path
import cv2
import numpy as np
from IPython.display import display, Image as IPImage

from landmarkdiff.landmarks import extract_landmarks, load_image
from landmarkdiff.manipulation import apply_procedure_preset, PROCEDURE_LANDMARKS
from landmarkdiff.masking import generate_surgical_mask
from landmarkdiff.inference import mask_composite
from landmarkdiff.synthetic.tps_warp import warp_image_tps

## 1. Load images from a directory

Set `INPUT_DIR` to a folder containing face images.

In [ ]:
INPUT_DIR = Path("sample_faces")  # Change to your image directory
OUTPUT_DIR = Path("batch_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_paths = sorted(
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in EXTENSIONS
)
print(f"Found {len(image_paths)} images")
for p in image_paths[:5]:
    print(f"  {p.name}")
if len(image_paths) > 5:
    print(f"  ... and {len(image_paths) - 5} more")

## 2. Extract landmarks from all images

This step detects faces and extracts 478 MediaPipe landmarks per image.
Images where no face is detected are skipped.

In [ ]:
results = []  # List of (path, image, landmarks)
skipped = []

for path in image_paths:
    img = load_image(str(path))
    face = extract_landmarks(img)
    if face is None:
        skipped.append(path.name)
        continue
    results.append((path, img, face))

print(f"Extracted landmarks: {len(results)} / {len(image_paths)}")
if skipped:
    print(f"Skipped (no face detected): {skipped}")

## 3. Batch process: apply procedure to all images

Apply a single procedure at a fixed intensity using TPS mode (CPU, no GPU required).

In [ ]:
PROCEDURE = "rhinoplasty"
INTENSITY = 60.0

predictions = []

for path, img, face in results:
    # Apply deformation
    modified = apply_procedure_preset(face, PROCEDURE, INTENSITY)
    
    # Warp image using TPS (CPU-based, no diffusion model needed)
    warped = warp_image_tps(img, face, modified)
    
    # Generate mask and composite
    mask = generate_surgical_mask(face, PROCEDURE, img.shape[:2])
    composited = mask_composite(warped, img, mask)
    
    # Save result
    out_path = OUTPUT_DIR / f"{path.stem}_{PROCEDURE}.png"
    cv2.imwrite(str(out_path), composited)
    predictions.append((path.name, out_path, composited))

print(f"Processed {len(predictions)} images -> {OUTPUT_DIR}/")

## 4. Multi-procedure batch

Apply different procedures and intensities to each image.

In [ ]:
# Define procedure configurations per image or globally
procedure_configs = [
    {"procedure": "rhinoplasty", "intensity": 50.0},
    {"procedure": "blepharoplasty", "intensity": 40.0},
    {"procedure": "rhytidectomy", "intensity": 60.0},
]

for path, img, face in results:
    for config in procedure_configs:
        proc = config["procedure"]
        intensity = config["intensity"]
        
        modified = apply_procedure_preset(face, proc, intensity)
        warped = warp_image_tps(img, face, modified)
        mask = generate_surgical_mask(face, proc, img.shape[:2])
        composited = mask_composite(warped, img, mask)
        
        out_path = OUTPUT_DIR / f"{path.stem}_{proc}_i{int(intensity)}.png"
        cv2.imwrite(str(out_path), composited)

total = len(results) * len(procedure_configs)
print(f"Generated {total} predictions ({len(results)} images x {len(procedure_configs)} procedures)")

## 5. Intensity sweep

Generate predictions at multiple intensity levels for a single image.

In [ ]:
if results:
    path, img, face = results[0]  # Use the first image
    proc = "rhinoplasty"
    intensities = [20, 40, 60, 80, 100]
    
    sweep_dir = OUTPUT_DIR / "intensity_sweep"
    sweep_dir.mkdir(exist_ok=True)
    
    for intensity in intensities:
        modified = apply_procedure_preset(face, proc, float(intensity))
        warped = warp_image_tps(img, face, modified)
        mask = generate_surgical_mask(face, proc, img.shape[:2])
        composited = mask_composite(warped, img, mask)
        
        out_path = sweep_dir / f"{path.stem}_{proc}_i{intensity}.png"
        cv2.imwrite(str(out_path), composited)
    
    print(f"Intensity sweep saved to {sweep_dir}/")
else:
    print("No images loaded -- set INPUT_DIR above")

## 6. Collect statistics

Compute landmark displacement statistics across the batch.

In [ ]:
proc = "rhinoplasty"
intensity = 60.0

displacements = []
for path, img, face in results:
    modified = apply_procedure_preset(face, proc, intensity)
    orig_px = face.pixel_coords[:, :2]
    mod_px = modified.pixel_coords[:, :2]
    disp = np.sqrt(np.sum((mod_px - orig_px) ** 2, axis=1))
    displacements.append(disp)

if displacements:
    all_disp = np.stack(displacements)
    print(f"Batch displacement statistics ({len(results)} images):")
    print(f"  Mean displacement: {all_disp.mean():.2f} px")
    print(f"  Max displacement:  {all_disp.max():.2f} px")
    print(f"  Std deviation:     {all_disp.std():.2f} px")
    
    # Per-procedure-region stats
    proc_indices = PROCEDURE_LANDMARKS[proc]
    region_disp = all_disp[:, proc_indices]
    print(f"  Region mean:       {region_disp.mean():.2f} px")
    print(f"  Region max:        {region_disp.max():.2f} px")
else:
    print("No images loaded")

## 7. Export results as CSV

Save per-image metadata and displacement statistics to a CSV file.

In [ ]:
import csv

csv_path = OUTPUT_DIR / "batch_results.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "width", "height", "confidence",
                     "procedure", "intensity", "mean_displacement", "max_displacement"])
    
    for (path, img, face), disp in zip(results, displacements):
        writer.writerow([
            path.name,
            face.image_width,
            face.image_height,
            f"{face.confidence:.3f}",
            proc,
            intensity,
            f"{disp.mean():.2f}",
            f"{disp.max():.2f}",
        ])

print(f"Results saved to {csv_path}")

## Available Procedures

LandmarkDiff supports the following surgical procedure presets:

In [ ]:
print("Available procedures:")
for proc in sorted(PROCEDURE_LANDMARKS.keys()):
    n_landmarks = len(PROCEDURE_LANDMARKS[proc])
    print(f"  {proc:30s} ({n_landmarks} landmarks)")